# ИССЛЕДОВАНИЕ ГЕНДЕРНОГО РАЗРЫВА В ЗАРАБОТНЫХ ПЛАТАХ
## На данных RLMS за 2010 и 2020 годы (Москва и Брянск)

**Автор:** Козлов Кирилл  
**Учебное заведение:** РАНХиГС, Цифровая Экономика, ИЭМИТ, 3 курс  
**Дата:** 2025


## 1. ИМПОРТ БИБЛИОТЕК


In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

# Эконометрические библиотеки
import statsmodels.api as sm
from statsmodels.stats.diagnostic import het_breuschpagan, het_white
from statsmodels.stats.stattools import durbin_watson
from statsmodels.stats.outliers_influence import variance_inflation_factor
from scipy import stats

# Настройка отображения
plt.style.use('seaborn-v0_8')
sns.set_palette("husl")
pd.set_option('display.max_columns', None)
pd.set_option('display.max_rows', 100)

# Для корректного отображения русского текста в графиках
plt.rcParams['font.family'] = 'DejaVu Sans'


## 2. ЗАГРУЗКА ДАННЫХ

**Регионы:** Москва и Тула  
**Годы:** 2010 и 2020

Загружаем данные RLMS по индивидам (RLMS_IND) и фильтруем по нужным годам и регионам.


In [ ]:
# Загрузка данных RLMS по индивидам
# Используем файл RLMS_IND_1994_2024_v3_rus.dta

print("Загрузка данных из файла RLMS_IND_1994_2024_v3_rus.dta...")
# convert_categoricals=False чтобы избежать проблем с категориальными переменными
df_full = pd.read_stata('data/RLMS_IND_1994_2024_v3_rus.dta', convert_categoricals=False)
print(f"Данные загружены. Размер: {df_full.shape}")

# Просмотр структуры данных
print("\nПервые строки данных:")
print(df_full[['idind', 'year', 'region', 'h5']].head(10))

# Проверка наличия нужных переменных
print("\nПроверка ключевых переменных:")
key_vars = ['year', 'region', 'h5', 'j13.2', 'j13.3', 'age', 'j1', 'j7', 'marst']
for var in key_vars:
    if var in df_full.columns:
        print(f"  OK: {var}")
    else:
        print(f"  НЕТ: {var}")

# Проверка годов и регионов
print(f"\nДоступные годы: {sorted(df_full['year'].dropna().unique())}")
print(f"Количество уникальных регионов: {df_full['region'].nunique()}")
print(f"Примеры регионов: {df_full['region'].dropna().unique()[:10]}")


## 3. ПОДГОТОВКА ДАННЫХ

### 3.1. Фильтрация по годам и регионам

Фильтруем данные по:
- **Годы:** 2010 и 2020
- **Регионы:** Москва (код региона нужно найти) и Тула (код региона нужно найти)

### 3.2. Создание переменных

Создаём переменные для анализа:
- `female` - дамми-переменная пола (1 - женщина, 0 - мужчина)
- `ln_wage` - логарифм заработной платы
- `age` и `age_sq` - возраст и квадрат возраста
- `education` - образование
- `experience` и `experience_sq` - опыт работы и квадрат опыта
- `married` - семейное положение
- `moscow` - дамми для Москвы
- `year_2020` - дамми для 2020 года


In [ ]:
# Фильтрация по годам (2010 и 2020)
print("Фильтрация по годам 2010 и 2020...")
df = df_full[df_full['year'].isin([2010, 2020])].copy()
print(f"Размер после фильтрации по годам: {df.shape}")

# Поиск кодов регионов для Москвы и Тулы
print("\nПоиск кодов регионов Москва и Тула...")
print("Уникальные регионы в данных:")
unique_regions = df['region'].dropna().unique()
print(f"Всего уникальных регионов: {len(unique_regions)}")

# Попробуем найти Москву и Тулу по названию или коду
# Обычно Москва имеет код 77 или название содержит "Москва"
# Тула имеет код 71 или название содержит "Тула"
moscow_codes = []
tula_codes = []

# Проверяем числовые коды
if df['region'].dtype in ['int64', 'float64']:
    # Москва обычно код 77, Тула - 71
    if 77 in unique_regions:
        moscow_codes.append(77)
    if 71 in unique_regions:
        tula_codes.append(71)
    # Также проверяем другие возможные коды
    for code in unique_regions:
        if isinstance(code, (int, float)) and not pd.isna(code):
            if 70 <= code <= 79:  # Центральный федеральный округ
                if code not in moscow_codes and code not in tula_codes:
                    print(f"  Найден регион с кодом {code}")

# Если регион - строковый, ищем по названию
if df['region'].dtype == 'object':
    for region in unique_regions[:50]:  # Проверяем первые 50
        if isinstance(region, str):
            region_lower = region.lower()
            if 'москв' in region_lower:
                moscow_codes.append(region)
            if 'тул' in region_lower:
                tula_codes.append(region)

print(f"\nНайденные коды для Москвы: {moscow_codes}")
print(f"Найденные коды для Тулы: {tula_codes}")

# Если не нашли автоматически, используем первые два региона из данных
if not moscow_codes or not tula_codes:
    print("\nНе удалось автоматически определить коды. Используем первые два региона.")
    sample_regions = df['region'].dropna().unique()[:2]
    moscow_codes = [sample_regions[0]]
    tula_codes = [sample_regions[1]]
    print(f"Используем для Москвы: {moscow_codes[0]}")
    print(f"Используем для Тулы: {tula_codes[0]}")

# Фильтрация по регионам
df = df[df['region'].isin(moscow_codes + tula_codes)].copy()
print(f"\nРазмер после фильтрации по регионам: {df.shape}")

# Создание переменных
print("\nСоздание переменных...")

# 1. Дамми-переменная пола (h5: 1 - мужчина, 2 - женщина)
if 'h5' in df.columns:
    df['female'] = (df['h5'] == 2).astype(int)
    print(f"  OK: female создана. Распределение: {df['female'].value_counts().to_dict()}")
else:
    print("  ОШИБКА: переменная h5 (пол) не найдена")

# 2. Логарифм заработной платы
# Пробуем разные варианты названий переменных зарплаты
wage_vars = ['j13.2', 'j13_2', 'j132', 'j13.3', 'j13_3', 'j133', 'wage', 'salary']
wage_var = None
for var in wage_vars:
    if var in df.columns:
        wage_var = var
        break

if wage_var:
    # Исключаем нулевые, отрицательные и пропущенные значения
    df_wage = df[df[wage_var] > 0].copy()
    df_wage['ln_wage'] = np.log(df_wage[wage_var])
    df = df_wage.copy()
    print(f"  OK: ln_wage создана из переменной {wage_var}")
    print(f"      Количество наблюдений с зарплатой: {len(df)}")
else:
    print("  ПРЕДУПРЕЖДЕНИЕ: переменная заработной платы не найдена")
    print(f"      Доступные переменные с 'j13': {[col for col in df.columns if 'j13' in str(col).lower()]}")

# 3. Возраст и квадрат возраста
if 'age' in df.columns:
    df['age_sq'] = df['age'] ** 2
    print(f"  OK: age и age_sq созданы")
else:
    # Пробуем вычислить из года рождения
    birth_vars = ['h4_1_y', 'birth_year', 'byear', 'ybirth']
    birth_var = None
    for var in birth_vars:
        if var in df.columns:
            birth_var = var
            break
    
    if birth_var:
        df['age'] = df['year'] - df[birth_var]
        df['age_sq'] = df['age'] ** 2
        print(f"  OK: age вычислен из {birth_var}")
    else:
        print("  ПРЕДУПРЕЖДЕНИЕ: переменная возраста не найдена")

# 4. Образование
if 'educ' in df.columns:
    # Используем educ напрямую или преобразуем
    df['education'] = df['educ'].astype(float)
    print(f"  OK: education создана из educ")
elif 'j1' in df.columns:
    # Преобразование категорий образования в годы (пример)
    education_map = {1: 9, 2: 11, 3: 13, 4: 15, 5: 17}
    df['education'] = df['j1'].map(education_map)
    print(f"  OK: education создана из j1")
else:
    print("  ПРЕДУПРЕЖДЕНИЕ: переменная образования не найдена")

# 5. Опыт работы
if 'j7' in df.columns:
    df['experience'] = df['j7'].astype(float)
    df['experience_sq'] = df['experience'] ** 2
    print(f"  OK: experience создана из j7")
else:
    print("  ПРЕДУПРЕЖДЕНИЕ: переменная опыта работы не найдена")

# 6. Семейное положение
if 'marst' in df.columns:
    # marst: 1 - женат/замужем, другие - нет
    df['married'] = (df['marst'] == 1).astype(int)
    print(f"  OK: married создана из marst")
else:
    print("  ПРЕДУПРЕЖДЕНИЕ: переменная семейного положения не найдена")

# 7. Дамми-переменные для региона и года
df['moscow'] = (df['region'].isin(moscow_codes)).astype(int)
df['year_2020'] = (df['year'] == 2020).astype(int)
print(f"  OK: moscow и year_2020 созданы")

# 8. Исключение пропущенных значений для ключевых переменных
variables_needed = ['ln_wage', 'female', 'age', 'age_sq']
variables_optional = ['education', 'experience', 'experience_sq', 'married']
variables_all = [v for v in variables_needed + variables_optional if v in df.columns]

print(f"\nИсключение пропущенных значений для переменных: {variables_all}")
df_clean = df[variables_all + ['moscow', 'year_2020', 'region', 'year']].copy()
df_clean = df_clean.dropna(subset=variables_needed)
print(f"Размер после исключения пропущенных: {df_clean.shape}")

# 9. Ограничение выборки по возрасту (18-65 лет)
if 'age' in df_clean.columns:
    df_clean = df_clean[(df_clean['age'] >= 18) & (df_clean['age'] <= 65)]
    print(f"Размер после ограничения по возрасту: {df_clean.shape}")

df = df_clean.copy()

# Итоговая статистика
print("\n" + "="*70)
print("ИТОГОВАЯ СТАТИСТИКА ПО ПОДГОТОВЛЕННЫМ ДАННЫМ")
print("="*70)
print(f"Размер итоговой выборки: {len(df)} наблюдений")
print(f"\nРаспределение по полу:")
print(df['female'].value_counts())
print(f"\nРаспределение по регионам:")
print(df['region'].value_counts())
print(f"\nРаспределение по годам:")
print(df['year'].value_counts())
print(f"\nРаспределение по полу и региону:")
print(pd.crosstab(df['female'], df['moscow'], margins=True))
print(f"\nРаспределение по полу и году:")
print(pd.crosstab(df['female'], df['year_2020'], margins=True))


## 4. ОПИСАТЕЛЬНАЯ СТАТИСТИКА


In [ ]:
# ОПИСАТЕЛЬНАЯ СТАТИСТИКА

print("="*70)
print("ОПИСАТЕЛЬНАЯ СТАТИСТИКА")
print("="*70)

# Основные статистики по всем переменным
vars_for_desc = ['ln_wage', 'female', 'age']
if 'education' in df.columns:
    vars_for_desc.append('education')
if 'experience' in df.columns:
    vars_for_desc.append('experience')
if 'married' in df.columns:
    vars_for_desc.append('married')

print("\nОсновные статистики:")
print(df[vars_for_desc].describe())

# Сравнение средних заработных плат
print("\n" + "="*70)
print("СРАВНЕНИЕ СРЕДНИХ ЗАРАБОТНЫХ ПЛАТ ПО ПОЛУ")
print("="*70)
wage_by_gender = df.groupby('female')['ln_wage'].agg(['mean', 'std', 'count'])
wage_by_gender.index = ['Мужчины', 'Женщины']
print(wage_by_gender)

# Разница в средних
wage_diff = wage_by_gender.loc['Мужчины', 'mean'] - wage_by_gender.loc['Женщины', 'mean']
wage_diff_pct = (wage_diff / wage_by_gender.loc['Мужчины', 'mean']) * 100
print(f"\nРазница в средних (логарифмах): {wage_diff:.4f}")
print(f"Разница в процентах: {wage_diff_pct:.2f}%")

# t-тест для сравнения средних
male_wages = df[df['female'] == 0]['ln_wage']
female_wages = df[df['female'] == 1]['ln_wage']
t_stat, p_value = stats.ttest_ind(male_wages, female_wages)
print(f"\nt-тест для сравнения средних:")
print(f"  t-статистика: {t_stat:.4f}")
print(f"  p-value: {p_value:.4f}")
if p_value < 0.05:
    print(f"  Вывод: статистически значимая разница (p < 0.05)")
else:
    print(f"  Вывод: статистически значимой разницы нет (p >= 0.05)")

# Сравнение по регионам и годам
print("\n" + "="*70)
print("СРЕДНИЕ ЗАРАБОТНЫЕ ПЛАТЫ ПО ПОЛУ, РЕГИОНУ И ГОДУ")
print("="*70)
wage_summary = df.groupby(['female', 'moscow', 'year_2020'])['ln_wage'].mean().unstack(level=[1, 2])
wage_summary.index = ['Мужчины', 'Женщины']
wage_summary.columns = ['Тула 2010', 'Тула 2020', 'Москва 2010', 'Москва 2020']
print(wage_summary)

# Дополнительная статистика по группам
print("\n" + "="*70)
print("ДОПОЛНИТЕЛЬНАЯ СТАТИСТИКА")
print("="*70)
print("\nСредние заработные платы по полу и региону:")
print(df.groupby(['female', 'moscow'])['ln_wage'].mean().unstack())

print("\nСредние заработные платы по полу и году:")
print(df.groupby(['female', 'year_2020'])['ln_wage'].mean().unstack())


## 5. ОЦЕНКА ЭКОНОМЕТРИЧЕСКИХ МОДЕЛЕЙ

### 5.1. Модель 1: Базовая модель

$$\ln(W_i) = \beta_0 + \beta_1 Female_i + \beta_2 Age_i + \beta_3 Age_i^2 + \beta_4 Education_i + \beta_5 Experience_i + \beta_6 Experience_i^2 + u_i$$


In [ ]:
# МОДЕЛЬ 1: Базовая модель

# Определение переменных для базовой модели
vars_model1 = ['female', 'age', 'age_sq']
if 'education' in df.columns:
    vars_model1.append('education')
if 'experience' in df.columns:
    vars_model1.append('experience')
if 'experience_sq' in df.columns:
    vars_model1.append('experience_sq')

X1 = df[vars_model1].copy()
X1 = sm.add_constant(X1)  # добавление константы
y = df['ln_wage']

# Оценка модели
model1 = sm.OLS(y, X1).fit()

# Вывод результатов
print("МОДЕЛЬ 1: Базовая модель")
print("=" * 70)
print(model1.summary())

# Сохранение результатов
try:
    results_summary = model1.summary().as_text()
    with open('results/model1_summary.txt', 'w', encoding='utf-8') as f:
        f.write(results_summary)
    print("\nРезультаты сохранены в results/model1_summary.txt")
except:
    print("\nНе удалось сохранить результаты (возможно, папка results не существует)")

# Интерпретация коэффициента female
beta_female = model1.params['female']
se_female = model1.bse['female']
pval_female = model1.pvalues['female']
print(f"\nКоэффициент female: {beta_female:.4f}")
print(f"Стандартная ошибка: {se_female:.4f}")
print(f"p-value: {pval_female:.4f}")
if pval_female < 0.05:
    print(f"Интерпретация: заработные платы женщин на {abs(beta_female)*100:.2f}% ниже заработных плат мужчин")
else:
    print("Интерпретация: статистически значимого гендерного разрыва не обнаружено")


### 5.2. Модель 2: С контролем семейного положения

$$\ln(W_i) = \beta_0 + \beta_1 Female_i + \beta_2 Age_i + \beta_3 Age_i^2 + \beta_4 Education_i + \beta_5 Experience_i + \beta_6 Experience_i^2 + \beta_7 Married_i + u_i$$


In [ ]:
# МОДЕЛЬ 2: С контролем семейного положения

vars_model2 = vars_model1.copy()
if 'married' in df.columns:
    vars_model2.append('married')

X2 = df[vars_model2].copy()
X2 = sm.add_constant(X2)
model2 = sm.OLS(y, X2).fit()

print("\n\nМОДЕЛЬ 2: С контролем семейного положения")
print("=" * 70)
print(model2.summary())

# Сравнение с моделью 1
print(f"\nСравнение с моделью 1:")
print(f"  Коэффициент female в модели 1: {model1.params['female']:.4f}")
print(f"  Коэффициент female в модели 2: {model2.params['female']:.4f}")
print(f"  Изменение: {model2.params['female'] - model1.params['female']:.4f}")
print(f"  R² модели 1: {model1.rsquared:.4f}")
print(f"  R² модели 2: {model2.rsquared:.4f}")


### 5.3. Модель 4: С контролем региона и года

$$\ln(W_i) = \beta_0 + \beta_1 Female_i + \beta_2 Age_i + \beta_3 Age_i^2 + \beta_4 Education_i + \beta_5 Experience_i + \beta_6 Experience_i^2 + \beta_7 Married_i + \beta_8 Moscow_i + \beta_9 Year2020_i + u_i$$


In [ ]:
# МОДЕЛЬ 4: С контролем региона и года

vars_model4 = vars_model2.copy()
vars_model4.extend(['moscow', 'year_2020'])

X4 = df[vars_model4].copy()
X4 = sm.add_constant(X4)
model4 = sm.OLS(y, X4).fit()

print("\n\nМОДЕЛЬ 4: С контролем региона и года")
print("=" * 70)
print(model4.summary())

# Интерпретация
print(f"\nИнтерпретация коэффициентов:")
print(f"  female: {model4.params['female']:.4f} (p={model4.pvalues['female']:.4f})")
print(f"  moscow: {model4.params['moscow']:.4f} (p={model4.pvalues['moscow']:.4f})")
print(f"  year_2020: {model4.params['year_2020']:.4f} (p={model4.pvalues['year_2020']:.4f})")


### 5.4. Модель 6: С взаимодействием пола и года

$$\ln(W_i) = \beta_0 + \beta_1 Female_i + \beta_2 Year2020_i + \beta_3 (Female_i \times Year2020_i) + \ldots + u_i$$

Коэффициент $\beta_3$ показывает, изменился ли гендерный разрыв во времени.


In [ ]:
# МОДЕЛЬ 6: С взаимодействием пола и года

# Создание переменной взаимодействия
df['female_year'] = df['female'] * df['year_2020']

vars_model6 = ['female', 'year_2020', 'female_year', 'age', 'age_sq']
if 'education' in df.columns:
    vars_model6.append('education')
if 'experience' in df.columns:
    vars_model6.append('experience')
if 'experience_sq' in df.columns:
    vars_model6.append('experience_sq')
if 'married' in df.columns:
    vars_model6.append('married')
vars_model6.append('moscow')

X6 = df[vars_model6].copy()
X6 = sm.add_constant(X6)
model6 = sm.OLS(y, X6).fit()

print("\n\nМОДЕЛЬ 6: С взаимодействием пола и года")
print("=" * 70)
print(model6.summary())

# Интерпретация взаимодействия
print(f"\nИнтерпретация взаимодействия:")
print(f"  Гендерный разрыв в 2010 году (female): {model6.params['female']:.4f}")
print(f"  Изменение разрыва к 2020 году (female_year): {model6.params['female_year']:.4f}")
print(f"  Гендерный разрыв в 2020 году: {model6.params['female'] + model6.params['female_year']:.4f}")
if model6.pvalues['female_year'] < 0.05:
    if model6.params['female_year'] > 0:
        print(f"  Вывод: гендерный разрыв СОКРАТИЛСЯ во времени (p={model6.pvalues['female_year']:.4f})")
    else:
        print(f"  Вывод: гендерный разрыв УВЕЛИЧИЛСЯ во времени (p={model6.pvalues['female_year']:.4f})")
else:
    print(f"  Вывод: изменение разрыва статистически незначимо (p={model6.pvalues['female_year']:.4f})")


## 6. ПРОВЕРКА ПРЕДПОСЫЛОК ТЕОРЕМЫ ГАУССА-МАРКОВА

### 6.1. Графики остатков


In [ ]:
# ПРОВЕРКА ПРЕДПОСЫЛОК ТЕОРЕМЫ ГАУССА-МАРКОВА
# Используем модель 4 (с контролем региона и года)

# Остатки и предсказанные значения
residuals = model4.resid
fitted_values = model4.fittedvalues

# График остатков vs предсказанные значения
fig, axes = plt.subplots(2, 2, figsize=(14, 10))

# 1. Остатки vs предсказанные значения
axes[0, 0].scatter(fitted_values, residuals, alpha=0.5, s=10)
axes[0, 0].axhline(y=0, color='r', linestyle='--', linewidth=2)
axes[0, 0].set_xlabel('Предсказанные значения')
axes[0, 0].set_ylabel('Остатки')
axes[0, 0].set_title('Остатки vs предсказанные значения')
axes[0, 0].grid(True, alpha=0.3)

# 2. Q-Q plot для проверки нормальности остатков
stats.probplot(residuals, dist="norm", plot=axes[0, 1])
axes[0, 1].set_title('Q-Q plot остатков')
axes[0, 1].grid(True, alpha=0.3)

# 3. Гистограмма остатков
axes[1, 0].hist(residuals, bins=50, edgecolor='black', alpha=0.7)
axes[1, 0].set_xlabel('Остатки')
axes[1, 0].set_ylabel('Частота')
axes[1, 0].set_title('Распределение остатков')
axes[1, 0].grid(True, alpha=0.3)

# 4. График остатков по порядку наблюдений
sample_size = min(1000, len(residuals))  # Берем выборку для визуализации
axes[1, 1].plot(residuals.iloc[:sample_size], alpha=0.7)
axes[1, 1].axhline(y=0, color='r', linestyle='--', linewidth=2)
axes[1, 1].set_xlabel('Номер наблюдения')
axes[1, 1].set_ylabel('Остатки')
axes[1, 1].set_title(f'Остатки по порядку наблюдений (первые {sample_size})')
axes[1, 1].grid(True, alpha=0.3)

plt.tight_layout()
try:
    plt.savefig('figures/residuals_diagnostics.png', dpi=300, bbox_inches='tight')
    print("График сохранен в figures/residuals_diagnostics.png")
except:
    print("Не удалось сохранить график (возможно, папка figures не существует)")
plt.show()


### 6.2. Тесты на гетероскедастичность


In [ ]:
# Тест Бройша-Пагана на гетероскедастичность
try:
    bp_test = het_breuschpagan(residuals, X4)
    print("ТЕСТ БРОЙША-ПАГАНА НА ГЕТЕРОСКЕДАСТИЧНОСТЬ")
    print("=" * 70)
    print(f"LM статистика: {bp_test[0]:.4f}")
    print(f"p-value: {bp_test[1]:.4f}")
    if bp_test[1] < 0.05:
        print("Вывод: отклоняем H0, присутствует гетероскедастичность")
        print("Рекомендация: использовать робастные стандартные ошибки")
    else:
        print("Вывод: не отклоняем H0, гомоскедастичность")
except Exception as e:
    print(f"Ошибка при выполнении теста Бройша-Пагана: {e}")

# Тест Уайта на гетероскедастичность
try:
    white_test = het_white(residuals, X4)
    print("\n" + "=" * 70)
    print("ТЕСТ УАЙТА НА ГЕТЕРОСКЕДАСТИЧНОСТЬ")
    print("=" * 70)
    print(f"LM статистика: {white_test[0]:.4f}")
    print(f"p-value: {white_test[1]:.4f}")
    if white_test[1] < 0.05:
        print("Вывод: отклоняем H0, присутствует гетероскедастичность")
        print("Рекомендация: использовать робастные стандартные ошибки")
    else:
        print("Вывод: не отклоняем H0, гомоскедастичность")
except Exception as e:
    print(f"Ошибка при выполнении теста Уайта: {e}")

# Если обнаружена гетероскедастичность, оцениваем модель с робастными ошибками
if 'bp_test' in locals() and bp_test[1] < 0.05:
    print("\n" + "=" * 70)
    print("ОЦЕНКА МОДЕЛИ С РОБАСТНЫМИ СТАНДАРТНЫМИ ОШИБКАМИ")
    print("=" * 70)
    model4_robust = sm.OLS(y, X4).fit(cov_type='HC3')
    print(model4_robust.summary())
    print(f"\nКоэффициент female с робастными ошибками: {model4_robust.params['female']:.4f}")
    print(f"Стандартная ошибка: {model4_robust.bse['female']:.4f}")
    print(f"p-value: {model4_robust.pvalues['female']:.4f}")


### 6.3. Проверка мультиколлинеарности (VIF)


### 6.4. Тест Рамсея на правильность спецификации модели


In [ ]:
# Тест Рамсея (RESET) на правильность спецификации модели
# Проверяет, нет ли пропущенных переменных или неправильной функциональной формы

from statsmodels.stats.diagnostic import linear_reset

try:
    reset_test = linear_reset(model4, power=2, test_type='fitted')
    print("ТЕСТ РАМСЕЯ (RESET) НА ПРАВИЛЬНОСТЬ СПЕЦИФИКАЦИИ МОДЕЛИ")
    print("=" * 70)
    print(f"F-статистика: {reset_test.fvalue:.4f}")
    print(f"p-value: {reset_test.pvalue:.4f}")
    if reset_test.pvalue < 0.05:
        print("Вывод: отклоняем H0, модель неправильно специфицирована")
        print("Рекомендация: возможно, нужно добавить дополнительные переменные или изменить функциональную форму")
    else:
        print("Вывод: не отклоняем H0, модель правильно специфицирована")
except Exception as e:
    print(f"Ошибка при выполнении теста Рамсея: {e}")
    print("Возможная причина: проблемы с данными или спецификацией модели")


### 6.5. Тест на нормальность остатков


In [ ]:
# Тест на нормальность остатков
# Тест Жака-Бера (для больших выборок) и тест Шапиро-Уилка (для малых выборок)

print("ТЕСТЫ НА НОРМАЛЬНОСТЬ ОСТАТКОВ")
print("=" * 70)

# Тест Жака-Бера
jb_stat, jb_p = stats.jarque_bera(residuals)
print(f"\nТест Жака-Бера:")
print(f"  Статистика: {jb_stat:.4f}")
print(f"  p-value: {jb_p:.4f}")
if jb_p < 0.05:
    print("  Вывод: отклоняем H0, остатки не распределены нормально")
else:
    print("  Вывод: не отклоняем H0, остатки распределены нормально")

# Тест Шапиро-Уилка (для выборок < 5000)
if len(residuals) < 5000:
    shapiro_stat, shapiro_p = stats.shapiro(residuals)
    print(f"\nТест Шапиро-Уилка:")
    print(f"  Статистика: {shapiro_stat:.4f}")
    print(f"  p-value: {shapiro_p:.4f}")
    if shapiro_p < 0.05:
        print("  Вывод: отклоняем H0, остатки не распределены нормально")
    else:
        print("  Вывод: не отклоняем H0, остатки распределены нормально")
else:
    print(f"\nТест Шапиро-Уилка не выполнен (выборка слишком большая: {len(residuals)} наблюдений)")
    print("Используйте тест Жака-Бера для больших выборок")


In [ ]:
# VIF (Variance Inflation Factor) для проверки мультиколлинеарности
print("ПРОВЕРКА МУЛЬТИКОЛЛИНЕАРНОСТИ (VIF)")
print("=" * 70)

try:
    vif_data = pd.DataFrame()
    vif_data['Переменная'] = X4.columns
    vif_data['VIF'] = [variance_inflation_factor(X4.values, i) for i in range(X4.shape[1])]
    print(vif_data.to_string(index=False))
    
    # Проверка на проблему мультиколлинеарности
    high_vif = vif_data[vif_data['VIF'] > 10]
    if len(high_vif) > 0:
        print(f"\nПРЕДУПРЕЖДЕНИЕ: Найдены переменные с VIF > 10:")
        print(high_vif.to_string(index=False))
        print("Это может указывать на проблему мультиколлинеарности")
    else:
        print("\nПроблем мультиколлинеарности не обнаружено (все VIF < 10)")
except Exception as e:
    print(f"Ошибка при расчете VIF: {e}")
    print("Возможная причина: слишком много пропущенных значений или проблемы с данными")


## 7. ИНТЕРПРЕТАЦИЯ РЕЗУЛЬТАТОВ

### Пример интерпретации коэффициента female

Если коэффициент $\beta_1 = -0.15$ и статистически значим:
- Заработные платы женщин на 15% ниже заработных плат мужчин при прочих равных условиях
- Это "необъяснённый" разрыв, который может быть связан с дискриминацией или ненаблюдаемыми характеристиками


In [ ]:
# ИТОГОВАЯ ИНТЕРПРЕТАЦИЯ РЕЗУЛЬТАТОВ

print("="*70)
print("ИТОГОВАЯ ИНТЕРПРЕТАЦИЯ РЕЗУЛЬТАТОВ")
print("="*70)

# Используем модель 4 как основную
beta_female = model4.params['female']
se_female = model4.bse['female']
pval_female = model4.pvalues['female']

print(f"\nКоэффициент female (модель 4): {beta_female:.4f}")
print(f"Стандартная ошибка: {se_female:.4f}")
print(f"p-value: {pval_female:.4f}")

print(f"\nИнтерпретация:")
if pval_female < 0.05:
    print(f"✓ Статистически значимый гендерный разрыв обнаружен (p < 0.05)")
    print(f"✓ Заработные платы женщин на {abs(beta_female)*100:.2f}% ниже заработных плат мужчин")
    print(f"  при прочих равных условиях (после учёта возраста, образования, опыта, региона и года).")
else:
    print("✗ Статистически значимого гендерного разрыва не обнаружено (p >= 0.05)")

# Доверительный интервал (95%)
ci_lower = beta_female - 1.96 * se_female
ci_upper = beta_female + 1.96 * se_female
print(f"\n95% доверительный интервал для коэффициента female:")
print(f"  [{ci_lower:.4f}, {ci_upper:.4f}]")
print(f"  В процентах: [{ci_lower*100:.2f}%, {ci_upper*100:.2f}%]")

# Сравнение моделей
print(f"\n" + "="*70)
print("СРАВНЕНИЕ МОДЕЛЕЙ")
print("="*70)
comparison = pd.DataFrame({
    'Модель': ['Модель 1', 'Модель 2', 'Модель 4', 'Модель 6'],
    'Коэффициент female': [
        model1.params['female'],
        model2.params['female'],
        model4.params['female'],
        model6.params['female']
    ],
    'Стандартная ошибка': [
        model1.bse['female'],
        model2.bse['female'],
        model4.bse['female'],
        model6.bse['female']
    ],
    'p-value': [
        model1.pvalues['female'],
        model2.pvalues['female'],
        model4.pvalues['female'],
        model6.pvalues['female']
    ],
    'R²': [
        model1.rsquared,
        model2.rsquared,
        model4.rsquared,
        model6.rsquared
    ],
    'Скорректированный R²': [
        model1.rsquared_adj,
        model2.rsquared_adj,
        model4.rsquared_adj,
        model6.rsquared_adj
    ],
    'AIC': [
        model1.aic,
        model2.aic,
        model4.aic,
        model6.aic
    ]
})
print(comparison.to_string(index=False))

# Выбор лучшей модели
print(f"\n" + "="*70)
print("ВЫБОР ЛУЧШЕЙ МОДЕЛИ")
print("="*70)
best_model_idx = comparison['Скорректированный R²'].idxmax()
best_model_name = comparison.loc[best_model_idx, 'Модель']
print(f"Лучшая модель по скорректированному R²: {best_model_name}")
print(f"  R² скорректированный: {comparison.loc[best_model_idx, 'Скорректированный R²']:.4f}")
print(f"  Коэффициент female: {comparison.loc[best_model_idx, 'Коэффициент female']:.4f}")
print(f"  p-value: {comparison.loc[best_model_idx, 'p-value']:.4f}")


## 8. ВЫВОДЫ

В этом разделе подведите итоги исследования:
1. Наличие/отсутствие гендерного разрыва
2. Величина разрыва
3. Динамика разрыва во времени
4. Региональные различия
5. Ограничения исследования
